# Identifiability at three levels of evidence

Profile likelihood (Section 3.5, Algorithm 2). One coordinate of the decision vector is fixed on a grid and the other 51 are re-optimised, against the measurements alone (D1), the measurements and the closures (D2) and the complete estimator (D3). The value is identifiable at a level when the profile of that level rises above the threshold of the likelihood-ratio test, chi-squared(1, 0.95) = 3.8415, on both sides. The complete computation (two feedstock variables, six nodes, three feedstocks) takes hours and is run with `python -m ccpyro.identifiability`; this notebook plots the profiles of the article and recomputes one of them.

In [ ]:
import os
import shutil
import sys
import tempfile
from pathlib import Path
ARTICLE = Path.cwd().parent / "results"
OUTPUT = Path.cwd() / "output" / "03_identifiability"
OUTPUT.mkdir(parents=True, exist_ok=True)
if (ARTICLE / "best_vectors.npz").exists():
    handle, temporary = tempfile.mkstemp(dir=OUTPUT, suffix=".npz")
    os.close(handle)
    shutil.copyfile(ARTICLE / "best_vectors.npz", temporary)
    os.replace(temporary, OUTPUT / "best_vectors.npz")
os.environ.setdefault("CCPYRO_RESULTS", str(OUTPUT))
os.environ.setdefault("CCPYRO_FIGDATA", str(OUTPUT / "figdata"))
try:
    import ccpyro  # installed with `pip install -e .`
except ImportError:  # running from a clone without installation
    sys.path.insert(0, str(Path.cwd().parent / "src"))
    import ccpyro
import numpy as np
import pandas as pd
np.set_printoptions(precision=6, suppress=True)
print("ccpyro", ccpyro.__version__)

In [ ]:
from ccpyro import model as mp, identifiability as idf
import matplotlib.pyplot as plt
curves = pd.read_csv(ARTICLE / "profile_curves.csv")
intervals = pd.read_csv(ARTICLE / "profile_intervals.csv")
cols = ["treatment", "variable", "T"] + [f"{lvl}_{b}" for lvl in ("measurements", "closures", "estimator") for b in ("lower", "upper")]
intervals[cols].round(3)

In [ ]:
def plot_profile(frame, title):
    fig, ax = plt.subplots(figsize=(5.5, 3.6))
    for tag, name in (("measurements", "D1 measurements"), ("closures", "D2 + closures"), ("estimator", "D3 estimator")):
        sub = frame[frame.criterion == tag].sort_values("value")
        ax.plot(sub.value, sub[tag] - sub[tag].min(), marker=".", label=name)
    ax.axhline(idf.CHI2_95_1DOF, color="k", ls="--", lw=1)
    ax.set_ylim(0, 4 * idf.CHI2_95_1DOF); ax.set_xlabel(title); ax.set_ylabel("increase of the criterion")
    ax.legend(fontsize=8)
    return ax

sel = curves[(curves.treatment == "rice husk (pelletized)") & (curves.variable == "y_t") & (curves["T"] == 400.0)]
plot_profile(sel, "tar O/C at 400 °C, rice husk (article)");

## Recomputing one profile

A coarser grid than in the article keeps the run to a few minutes.

In [ ]:
built = mp.build_cases(random_starts=0)
label = "rice husk (pelletized)"
ctx, ref = built["contexts"][label], built["states"][label].vector
frame = idf.profile(ctx, ref, "y_t", node=1, points=5)
print(idf.profile_interval(frame))
plot_profile(frame, "tar O/C at 400 °C, rice husk (recomputed)");